## Training

The purpose of this note book is to setup training for the ED Fast Track model

In [224]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import pickle
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR, OneCycleLR

In [ ]:
class MultiInputFNN(nn.Module):
    def __init__(self, 
                 diagnosis_emb_dim=768, 
                 medication_emb_dim=768, 
                 pmh_emb_dim=768,
                 emb_dense_dim=50,
                 raw_features_dim=100,  # Set based on your feature count
                 hidden_dims=[50, 50, 50, 2]):
        super(MultiInputFNN, self).__init__()
        
        # Process diagnosis embeddings
        self.diagnosis_dense = nn.Linear(diagnosis_emb_dim, emb_dense_dim)
        self.diagnosis_batch_norm = nn.BatchNorm1d(emb_dense_dim)
        
        # Process medication embeddings
        self.medication_dense = nn.Linear(medication_emb_dim, emb_dense_dim)
        self.medication_batch_norm = nn.BatchNorm1d(emb_dense_dim)
        
        # Process PMH embeddings
        self.pmh_dense = nn.Linear(pmh_emb_dim, emb_dense_dim)
        self.pmh_batch_norm = nn.BatchNorm1d(emb_dense_dim)
        
        # Combined input dimension for first dense layer after concatenation
        combined_emb_dim = emb_dense_dim * 3
        self.combined_input_dim = combined_emb_dim + raw_features_dim
        
        # Main network path
        self.dense1 = nn.Linear(self.combined_input_dim, hidden_dims[0])
        self.dense2 = nn.Linear(hidden_dims[0], hidden_dims[1])
        self.dense3 = nn.Linear(hidden_dims[1], hidden_dims[2])
        self.output = nn.Linear(hidden_dims[2], hidden_dims[3])
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, diagnosis_emb, medication_emb, pmh_emb, raw_features):
        # Process each embedding type
        x_diag = self.diagnosis_batch_norm(self.relu(self.diagnosis_dense(diagnosis_emb)))
        x_med = self.medication_batch_norm(self.relu(self.medication_dense(medication_emb)))
        x_pmh = self.pmh_batch_norm(self.relu(self.pmh_dense(pmh_emb)))
        
        # Concatenate embeddings with raw features
        x_combined = torch.cat([x_diag, x_med, x_pmh, raw_features], dim=1)
        
        # Forward pass through main network
        x = self.relu(self.dense1(x_combined))
        x = self.dropout(self.relu(self.dense2(x)))
        x = self.relu(self.dense3(x))
        x = self.output(x)
        
        return x

In [ ]:
class MultiInputDataset(Dataset):
    def __init__(self, diagnosis_embeddings, medication_embeddings, pmh_embeddings, features, labels=None):
        """
        Args:
            diagnosis_embeddings: Dictionary mapping CSN to diagnosis embeddings
            medication_embeddings: Dictionary mapping CSN to medication embeddings
            pmh_embeddings: Dictionary mapping CSN to PMH embeddings
            features: DataFrame or array of features
            labels: Target labels
        """
        # Get CSNs from the features DataFrame
        self.csns = features.index.tolist() if isinstance(features, pd.DataFrame) else list(range(len(features)))
        self.features = features.values if isinstance(features, pd.DataFrame) else features
        
        # Store embeddings dictionaries
        self.diagnosis_embeddings = diagnosis_embeddings
        self.medication_embeddings = medication_embeddings
        self.pmh_embeddings = pmh_embeddings
        
        # Check if we have labels
        self.labels = labels.values if isinstance(labels, pd.DataFrame) else labels
        
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        # Get the CSN for this index
        csn = self.csns[idx]
        
        # Get embeddings for this CSN
        diagnosis_emb = torch.FloatTensor(self.diagnosis_embeddings[csn])
        medication_emb = torch.FloatTensor(self.medication_embeddings[csn])
        pmh_emb = torch.FloatTensor(self.pmh_embeddings[csn])
        
        # Get features
        feature = torch.FloatTensor(self.features[idx])
        
        if self.labels is not None:
            label = torch.LongTensor([self.labels[idx]]).squeeze() 
            return diagnosis_emb, medication_emb, pmh_emb, feature, label
        else:
            return diagnosis_emb, medication_emb, pmh_emb, feature

In [227]:
# Function to load data for a single split (train or validation)
def load_split_data(diag_emb_path, med_emb_path, pmh_emb_path, features_path, targets_path):
    # Load embeddings from pickle files
    with open(diag_emb_path, 'rb') as f:
        diagnosis_embeddings = pickle.load(f)
    
    with open(med_emb_path, 'rb') as f:
        medication_embeddings = pickle.load(f)
    
    with open(pmh_emb_path, 'rb') as f:
        pmh_embeddings = pickle.load(f)
    
    # Load features and targets from CSV
    features_df = pd.read_csv(features_path, index_col='CSN')  # Assuming CSN is in the features CSV
    targets_df = pd.read_csv(targets_path, index_col='CSN')  # Assuming CSN is in the targets CSV
    
    # Convert targets to numpy arrays
    targets = targets_df.values.flatten()
    
    return diagnosis_embeddings, medication_embeddings, pmh_embeddings, features_df, targets


In [ ]:
def get_scheduler(scheduler_name, optimizer, **kwargs):
    if scheduler_name == 'plateau':
        return ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10, **kwargs)
    elif scheduler_name == 'cosine':
        return CosineAnnealingLR(optimizer, T_max=100, **kwargs)
    elif scheduler_name == 'onecycle':
        return OneCycleLR(optimizer, max_lr=0.01, total_steps=100, **kwargs)
    else:
        return None

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler=None, scheduler_name='plateau', num_epochs=10, device='cuda'):
    model = model.to(device)
    best_val_loss = float('inf')
    best_val_auc = 0.0
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        for diagnosis_emb, medication_emb, pmh_emb, features, labels in train_loader:
            # print(f"Diagnosis Embedding Shape: {diagnosis_emb.shape}")
            # print(f"Medication Embedding Shape: {medication_emb.shape}")
            # print(f"PMH Embedding Shape: {pmh_emb.shape}")
            # print(f"Features Shape: {features.shape}")
            # print(f"Labels Shape: {labels.shape}")

            # Move data to device
            diagnosis_emb = diagnosis_emb.to(device)
            medication_emb = medication_emb.to(device)
            pmh_emb = pmh_emb.to(device)
            features = features.to(device)
            labels = labels.to(device)
            
            # Zero gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(diagnosis_emb, medication_emb, pmh_emb, features)
            loss = criterion(outputs, labels)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        all_labels = []
        all_probs = []
        
        with torch.no_grad():
            for diagnosis_emb, medication_emb, pmh_emb, features, labels in val_loader:
                diagnosis_emb = diagnosis_emb.to(device)
                medication_emb = medication_emb.to(device)
                pmh_emb = pmh_emb.to(device)
                features = features.to(device)
                labels = labels.to(device)
                
                outputs = model(diagnosis_emb, medication_emb, pmh_emb, features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                # Get probabilities
                # For binary classification
                probs = torch.softmax(outputs, dim=1)[:, 1] if outputs.shape[1] == 2 else torch.sigmoid(outputs)
                
                # Store for AUC calculation
                all_labels.append(labels.cpu().numpy())
                all_probs.append(probs.cpu().numpy())
        
        # Calculate average losses
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        
        # Calculate AUC ROC
        all_labels = np.concatenate(all_labels)
        all_probs = np.concatenate(all_probs)
        
        # For binary classification
        val_auc = roc_auc_score(all_labels, all_probs)
        
        # Update scheduler
        if scheduler is not None:
            if scheduler_name == 'plateau':
                scheduler.step(val_loss)
            else:
                scheduler.step()
        
        # Save best model - could save based on loss or AUC
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model_loss.pt')
            
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), 'best_model_auc.pt')
        
        print(f'Epoch {epoch+1}/{num_epochs}, '
              f'Train Loss: {train_loss:.4f}, '
              f'Val Loss: {val_loss:.4f}, '
              f'Val AUC: {val_auc:.4f}, '
              f'LR: {optimizer.param_groups[0]["lr"]:.6f}')
    
    return model

In [230]:
# Main execution code - with fixes
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

BERT_LIST = [
    'bert_base',
    'bio_clinical_bert',
    'blue_bert',
    'pubmed_bert'
]

SPLIT_TYPE = 'random'
BERT = BERT_LIST[1]

# Training data paths
train_diag_emb_path = f'data/embeddings/visit_diagnosis_{SPLIT_TYPE}_train_{BERT}.pkl'
train_med_emb_path = f'data/embeddings/meds_{SPLIT_TYPE}_train_{BERT}.pkl'
train_pmh_emb_path = f'data/embeddings/pmh_{SPLIT_TYPE}_train_{BERT}.pkl'
train_features_path = f'data/features_all_{SPLIT_TYPE}_train.csv'
train_targets_path = f'data/features_target_{SPLIT_TYPE}_train.csv'

# Validation data paths
val_diag_emb_path = f'data/embeddings/visit_diagnosis_{SPLIT_TYPE}_val_{BERT}.pkl'
val_med_emb_path = f'data/embeddings/meds_{SPLIT_TYPE}_val_{BERT}.pkl'
val_pmh_emb_path = f'data/embeddings/pmh_{SPLIT_TYPE}_val_{BERT}.pkl'
val_features_path = f'data/features_all_{SPLIT_TYPE}_val.csv'
val_targets_path = f'data/features_target_{SPLIT_TYPE}_val.csv'

# Load training data
train_diagnosis_embeddings, train_medication_embeddings, train_pmh_embeddings, train_features, train_targets = load_split_data(
    train_diag_emb_path, train_med_emb_path, train_pmh_emb_path, train_features_path, train_targets_path
)

# Load validation data
val_diagnosis_embeddings, val_medication_embeddings, val_pmh_embeddings, val_features, val_targets = load_split_data(
    val_diag_emb_path, val_med_emb_path, val_pmh_emb_path, val_features_path, val_targets_path
)

# Scale features - fit on training data, transform both train and validation
# scaler = StandardScaler()
# train_features_scaled = scaler.fit_transform(train_features)
# val_features_scaled = scaler.transform(val_features)

# Create datasets - adjusted for dictionary embeddings
train_dataset = MultiInputDataset(
    train_diagnosis_embeddings, 
    train_medication_embeddings, 
    train_pmh_embeddings, 
    train_features,
    train_targets
)

val_dataset = MultiInputDataset(
    val_diagnosis_embeddings, 
    val_medication_embeddings, 
    val_pmh_embeddings, 
    val_features,
    val_targets
)

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

# Get embedding dimensions from the first entry in each dictionary
# Fix for the error: 'dict' object has no attribute 'shape'
first_diag_key = next(iter(train_diagnosis_embeddings))
first_med_key = next(iter(train_medication_embeddings))
first_pmh_key = next(iter(train_pmh_embeddings))

diagnosis_emb_dim = len(train_diagnosis_embeddings[first_diag_key])
medication_emb_dim = len(train_medication_embeddings[first_med_key])
pmh_emb_dim = len(train_pmh_embeddings[first_pmh_key])

# Initialize model with correct dimensions
raw_features_dim = train_features.shape[1]
model = MultiInputFNN(
    diagnosis_emb_dim=diagnosis_emb_dim,
    medication_emb_dim=medication_emb_dim,
    pmh_emb_dim=pmh_emb_dim,
    raw_features_dim=raw_features_dim,
    emb_dense_dim=50,
    #hidden_dims=[50, 50, 50, 2]
    hidden_dims=[64, 128, 64, 2]  # Adjust based on your output dimension
)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Change based on your task (MSE for regression, CrossEntropy for classification)
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-5)

# LR Scheduler - choose one
scheduler_name = 'cosine'  # 'plateau', 'cosine', or 'onecycle'
scheduler = get_scheduler(
    scheduler_name, 
    optimizer,
)

# Train model
trained_model = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    scheduler_name=scheduler_name,
    num_epochs=13,
    device=device
)

print("Training complete!")

Using device: cpu
Epoch 1/13, Train Loss: 0.5171, Val Loss: 0.4621, Val AUC: 0.7838, LR: 0.000100
Epoch 2/13, Train Loss: 0.4584, Val Loss: 0.4437, Val AUC: 0.7988, LR: 0.000100
Epoch 3/13, Train Loss: 0.4465, Val Loss: 0.4337, Val AUC: 0.8103, LR: 0.000100
Epoch 4/13, Train Loss: 0.4367, Val Loss: 0.4342, Val AUC: 0.8111, LR: 0.000100
Epoch 5/13, Train Loss: 0.4328, Val Loss: 0.4311, Val AUC: 0.8113, LR: 0.000099
Epoch 6/13, Train Loss: 0.4298, Val Loss: 0.4266, Val AUC: 0.8199, LR: 0.000099
Epoch 7/13, Train Loss: 0.4255, Val Loss: 0.4279, Val AUC: 0.8177, LR: 0.000099
Epoch 8/13, Train Loss: 0.4240, Val Loss: 0.4290, Val AUC: 0.8162, LR: 0.000098
Epoch 9/13, Train Loss: 0.4229, Val Loss: 0.4303, Val AUC: 0.8156, LR: 0.000098
Epoch 10/13, Train Loss: 0.4202, Val Loss: 0.4225, Val AUC: 0.8221, LR: 0.000098
Epoch 11/13, Train Loss: 0.4178, Val Loss: 0.4281, Val AUC: 0.8140, LR: 0.000097
Epoch 12/13, Train Loss: 0.4149, Val Loss: 0.4205, Val AUC: 0.8234, LR: 0.000096
Epoch 13/13, Train 

In [231]:
# Test set evaluation
def evaluate_test_set(model_path, device='cuda'):
    # Load test data
    test_diag_emb_path = 'data/test_diagnosis_embeddings.pkl'
    test_med_emb_path = 'data/test_medication_embeddings.pkl'
    test_pmh_emb_path = 'data/test_pmh_embeddings.pkl'
    test_features_path = 'data/test_features.csv'
    test_targets_path = 'data/test_targets.csv'
    
    # Load test data using the same function
    test_diagnosis_embeddings, test_medication_embeddings, test_pmh_embeddings, test_features, test_targets = load_split_data(
        test_diag_emb_path, test_med_emb_path, test_pmh_emb_path, test_features_path, test_targets_path
    )
    
    # Scale features with the same scaler used for training
    test_features_scaled = scaler.transform(test_features)
    
    # Create test dataset and dataloader
    test_dataset = MultiInputDataset(
        test_diagnosis_embeddings, test_medication_embeddings, test_pmh_embeddings, 
        test_features_scaled, test_targets
    )
    test_loader = DataLoader(test_dataset, batch_size=32)
    
    # Load the best model
    best_model = MultiInputFNN(
        diagnosis_emb_dim=test_diagnosis_embeddings.shape[1],
        medication_emb_dim=test_medication_embeddings.shape[1],
        pmh_emb_dim=test_pmh_embeddings.shape[1],
        raw_features_dim=test_features_scaled.shape[1],
        hidden_dims=[50, 50, 50, 2]
    )
    best_model.load_state_dict(torch.load(model_path))
    best_model = best_model.to(device)
    
    # Evaluate
    best_model.eval()
    test_loss = 0.0
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for diagnosis_emb, medication_emb, pmh_emb, features, labels in test_loader:
            diagnosis_emb = diagnosis_emb.to(device)
            medication_emb = medication_emb.to(device)
            pmh_emb = pmh_emb.to(device)
            features = features.to(device)
            labels = labels.to(device)
            
            outputs = best_model(diagnosis_emb, medication_emb, pmh_emb, features)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            
            # Get probabilities
            probs = torch.softmax(outputs, dim=1)[:, 1] if outputs.shape[1] == 2 else torch.sigmoid(outputs)
            
            all_labels.append(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
    
    # Calculate metrics
    test_loss /= len(test_loader)
    all_labels = np.concatenate(all_labels)
    all_probs = np.concatenate(all_probs)
    test_auc = roc_auc_score(all_labels, all_probs)
    
    # Print results
    print(f'Test Loss: {test_loss:.4f}, Test AUC: {test_auc:.4f}')
    
    # Optionally, calculate threshold-based metrics
    threshold = 0.5
    predictions = (all_probs >= threshold).astype(int)
    
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    
    accuracy = accuracy_score(all_labels, predictions)
    precision = precision_score(all_labels, predictions)
    recall = recall_score(all_labels, predictions)
    f1 = f1_score(all_labels, predictions)
    
    print(f'Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}')
    
    return test_auc

# Run evaluation on both models (loss-based and AUC-based)
print("Evaluating model optimized for validation loss:")
loss_model_auc = evaluate_test_set('best_model_loss.pt', device)

print("\nEvaluating model optimized for validation AUC:")
auc_model_auc = evaluate_test_set('best_model_auc.pt', device)

print(f"\nBest test AUC: {max(loss_model_auc, auc_model_auc):.4f}")

Evaluating model optimized for validation loss:


FileNotFoundError: [Errno 2] No such file or directory: 'data/test_diagnosis_embeddings.pkl'